In [5]:
import os
import time
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)  

In [6]:
class Cfg:
    L, H = 2.2, 0.41
    xc, yc, R = 0.2, 0.2, 0.05
    rho, nu, Um = 1.0, 1e-3, 0.3

    # collocation counts
    n_pde   = 6000     # interior PDE points
    n_wake  = 2000     # extra points biased into the near-wake / boundary layer
    n_bc    = 400      # points per boundary segment (inlet, top, bottom, outlet)
    n_cyl   = 600      # points on the cylinder surface
    n_data  = 1500     # sparse FEM-supervision points

    # network
    width, depth = 64, 5      # matches the 2D-Burgers classical PINN capacity

    # loss weights
    w_pde, w_bc, w_data, w_gauge = 1.0, 10.0, 5.0, 1.0

    # optimisation
    adam_iters  = 2000
    adam_lr     = 1e-3
    lbfgs_iters = 0

    device = "cpu"     # like-for-like timing vs. QAPINN; nets are small
    seed   = 0

In [7]:
# GEOMETRY HELPERS
def _outside_cylinder(x, y, cfg, margin=0.0):
    return (x - cfg.xc) ** 2 + (y - cfg.yc) ** 2 >= (cfg.R + margin) ** 2


def sample_interior(n, cfg, rng, wake_bias=False):
    pts = []
    while len(pts) < n:
        m = n - len(pts)
        if wake_bias:
            x = rng.uniform(cfg.xc - cfg.R, cfg.xc + 8 * cfg.R, size=2 * m)
            y = rng.uniform(cfg.yc - 3 * cfg.R, cfg.yc + 3 * cfg.R, size=2 * m)
        else:
            x = rng.uniform(0, cfg.L, size=2 * m)
            y = rng.uniform(0, cfg.H, size=2 * m)
        keep = _outside_cylinder(x, y, cfg, margin=1e-4) & (x >= 0) & (x <= cfg.L) \
               & (y >= 0) & (y <= cfg.H)
        pts.extend(np.stack([x[keep], y[keep]], axis=1).tolist())
    return np.array(pts[:n])


def sample_boundaries(cfg, rng):
    n = cfg.n_bc
    # inlet (x=0): parabolic u, v=0
    y_in = rng.uniform(0, cfg.H, size=n)
    inlet_xy = np.stack([np.zeros(n), y_in], axis=1)
    u_in = 4 * cfg.Um * y_in * (cfg.H - y_in) / cfg.H ** 2
    inlet_uv = np.stack([u_in, np.zeros(n)], axis=1)

    # walls y=0 and y=H: no-slip
    x_w = rng.uniform(0, cfg.L, size=n)
    bot_xy = np.stack([x_w, np.zeros(n)], axis=1)
    top_xy = np.stack([x_w, np.full(n, cfg.H)], axis=1)
    wall_xy = np.concatenate([bot_xy, top_xy], axis=0)
    wall_uv = np.zeros_like(wall_xy)

    # outlet (x=L): pressure gauge p=0 (velocity left free / natural)
    y_out = rng.uniform(0, cfg.H, size=n)
    outlet_xy = np.stack([np.full(n, cfg.L), y_out], axis=1)

    # cylinder surface: no-slip
    th = rng.uniform(0, 2 * np.pi, size=cfg.n_cyl)
    cyl_xy = np.stack([cfg.xc + cfg.R * np.cos(th),
                       cfg.yc + cfg.R * np.sin(th)], axis=1)
    cyl_uv = np.zeros_like(cyl_xy)

    return dict(inlet_xy=inlet_xy, inlet_uv=inlet_uv,
                wall_xy=wall_xy, wall_uv=wall_uv,
                outlet_xy=outlet_xy,
                cyl_xy=cyl_xy, cyl_uv=cyl_uv)




In [8]:
# PINN
class PINN(nn.Module):
    def __init__(self, width=64, depth=5):
        super().__init__()
        layers, d_in = [], 2
        for _ in range(depth):
            lin = nn.Linear(d_in, width)
            nn.init.xavier_normal_(lin.weight); nn.init.zeros_(lin.bias)
            layers += [lin, nn.Tanh()]
            d_in = width
        out = nn.Linear(d_in, 3)          # u, v, p
        nn.init.xavier_normal_(out.weight); nn.init.zeros_(out.bias)
        layers.append(out)
        self.net = nn.Sequential(*layers)
        self._feat = None                 # last hidden layer, for XAI later

    def forward(self, xy):
        # intercept last hidden representation (for latent-space analysis)
        h = xy
        for layer in self.net[:-1]:
            h = layer(h)
        self._feat = h
        out = self.net[-1](h)
        return out[:, 0:1], out[:, 1:2], out[:, 2:3]   # u, v, p


In [9]:
## PDE residual via automatic differentiation

def navier_stokes_residual(model, xy, cfg):
    """Steady incompressible NS residuals at points xy = (x, y)."""
    xy = xy.clone().requires_grad_(True)
    u, v, p = model(xy)

    def grad(f):
        return torch.autograd.grad(f, xy, torch.ones_like(f),
                                   create_graph=True)[0]

    du, dv, dp = grad(u), grad(v), grad(p)
    u_x, u_y = du[:, 0:1], du[:, 1:2]
    v_x, v_y = dv[:, 0:1], dv[:, 1:2]
    p_x, p_y = dp[:, 0:1], dp[:, 1:2]

    u_xx = grad(u_x)[:, 0:1]; u_yy = grad(u_y)[:, 1:2]
    v_xx = grad(v_x)[:, 0:1]; v_yy = grad(v_y)[:, 1:2]

    # momentum-x, momentum-y, continuity
    r_u = u * u_x + v * u_y + p_x / cfg.rho - cfg.nu * (u_xx + u_yy)
    r_v = u * v_x + v * v_y + p_y / cfg.rho - cfg.nu * (v_xx + v_yy)
    r_c = u_x + v_y
    return r_u, r_v, r_c

In [10]:
## training  
def to_t(a, cfg):
    return torch.tensor(a, dtype=torch.float64, device=cfg.device)


def train(cfg=Cfg(), gt_path="gt_out/cylinder_gt.npz", outdir="pinn_out"):
    os.makedirs(outdir, exist_ok=True)
    torch.manual_seed(cfg.seed)
    rng = np.random.default_rng(cfg.seed)

    gt = None
    if os.path.exists(gt_path):
        gt = np.load(gt_path)
        cfg.L, cfg.H = float(gt["L"]), float(gt["H"])
        cfg.xc, cfg.yc, cfg.R = float(gt["xc"]), float(gt["yc"]), float(gt["R"])
        cfg.rho, cfg.nu, cfg.Um = float(gt["rho"]), float(gt["nu"]), float(gt["Um"])
        print(f"loaded ground truth: Re={float(gt['Re']):.1f}, "
              f"{len(gt['x'])} points")
    else:
        print("WARNING: no ground truth found; training PDE+BC only "
              "(run ground_truth.py first for data supervision & validation).")

    Xf   = to_t(np.concatenate([sample_interior(cfg.n_pde, cfg, rng),
                                sample_interior(cfg.n_wake, cfg, rng,
                                                wake_bias=True)]), cfg)
    B    = sample_boundaries(cfg, rng)
    Xin  = to_t(B["inlet_xy"], cfg);  Uin = to_t(B["inlet_uv"], cfg)
    Xw   = to_t(B["wall_xy"], cfg);   Uw  = to_t(B["wall_uv"], cfg)
    Xout = to_t(B["outlet_xy"], cfg)
    Xcyl = to_t(B["cyl_xy"], cfg);    Ucyl = to_t(B["cyl_uv"], cfg)

    # sparse data supervision from FEM ground truth
    if gt is not None:
        idx = rng.choice(len(gt["x"]), size=min(cfg.n_data, len(gt["x"])),
                         replace=False)
        Xd = to_t(np.stack([gt["x"][idx], gt["y"][idx]], axis=1), cfg)
        Ud = to_t(np.stack([gt["u"][idx], gt["v"][idx], gt["p"][idx]], axis=1), cfg)
    else:
        Xd = Ud = None

    model = PINN(cfg.width, cfg.depth).to(cfg.device)
    mse = nn.MSELoss()

    def closure_terms():
        # PDE
        r_u, r_v, r_c = navier_stokes_residual(model, Xf, cfg)
        L_pde = mse(r_u, torch.zeros_like(r_u)) \
              + mse(r_v, torch.zeros_like(r_v)) \
              + mse(r_c, torch.zeros_like(r_c))
        # BC: inlet, walls, cylinder (velocity); outlet pressure gauge
        ui, vi, _  = model(Xin)
        uw, vw, _  = model(Xw)
        uc, vc, _  = model(Xcyl)
        _,  _,  po = model(Xout)
        L_bc = mse(torch.cat([ui, vi], 1), Uin) \
             + mse(torch.cat([uw, vw], 1), Uw) \
             + mse(torch.cat([uc, vc], 1), Ucyl)
        L_gauge = mse(po, torch.zeros_like(po))
        # data
        if Xd is not None:
            ud, vd, pd = model(Xd)
            L_data = mse(torch.cat([ud, vd, pd], 1), Ud)
        else:
            L_data = torch.zeros((), device=cfg.device)
        return L_pde, L_bc, L_gauge, L_data

    def total_loss():
        L_pde, L_bc, L_gauge, L_data = closure_terms()
        return (cfg.w_pde * L_pde + cfg.w_bc * L_bc
                + cfg.w_gauge * L_gauge + cfg.w_data * L_data), \
               (L_pde, L_bc, L_gauge, L_data)

    history = {"loss": [], "pde": [], "bc": [], "data": []}

    # ---- Phase 1: Adam --------------------------------------------------------
    opt = torch.optim.Adam(model.parameters(), lr=cfg.adam_lr)
    t0 = time.time()
    for it in range(cfg.adam_iters):
        opt.zero_grad()
        loss, (Lp, Lb, Lg, Ld) = total_loss()
        loss.backward()
        opt.step()
        if it % 500 == 0 or it == cfg.adam_iters - 1:
            history["loss"].append(loss.item()); history["pde"].append(Lp.item())
            history["bc"].append(Lb.item());     history["data"].append(Ld.item())
            print(f"[adam {it:5d}] loss={loss.item():.3e}  "
                  f"pde={Lp.item():.2e} bc={Lb.item():.2e} "
                  f"gauge={Lg.item():.2e} data={Ld.item():.2e}")

    # ---- Phase 2: L-BFGS refinement ------------------------------------------
    opt = torch.optim.LBFGS(model.parameters(), max_iter=cfg.lbfgs_iters,
                            history_size=50, line_search_fn="strong_wolfe",
                            tolerance_grad=1e-12, tolerance_change=1e-14)

    def closure():
        opt.zero_grad()
        loss, _ = total_loss()
        loss.backward()
        return loss

    opt.step(closure)
    final, (Lp, Lb, Lg, Ld) = total_loss()
    train_time = time.time() - t0
    print(f"[lbfgs done] loss={final.item():.3e}  "
          f"pde={Lp.item():.2e} bc={Lb.item():.2e} data={Ld.item():.2e}  "
          f"({train_time:.1f}s)")

    # ---- validation against full ground truth --------------------------------
    rel_l2 = None
    if gt is not None:
        with torch.no_grad():
            Xall = to_t(np.stack([gt["x"], gt["y"]], axis=1), cfg)
            up, vp, pp = model(Xall)
            pred = torch.cat([up, vp], 1).cpu().numpy()
            true = np.stack([gt["u"], gt["v"]], axis=1)
            rel_l2 = np.linalg.norm(pred - true) / np.linalg.norm(true)
        print(f"velocity relative L2 vs FEM ground truth: {rel_l2:.4e}")

    # ---- save ----------------------------------------------------------------
    torch.save(model.state_dict(), os.path.join(outdir, "pinn_cylinder.pt"))
    np.savez(os.path.join(outdir, "history.npz"), **history,
             train_time=train_time, rel_l2=(rel_l2 or np.nan))
    _plot(model, gt, cfg, outdir)
    return model, dict(rel_l2=rel_l2, train_time=train_time,
                       final_loss=final.item())


def _plot(model, gt, cfg, outdir):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        if gt is None:
            return
        xy = np.stack([gt["x"], gt["y"]], axis=1)
        with torch.no_grad():
            up, vp, pp = model(to_t(xy, cfg))
        umag_pred = np.linalg.norm(np.concatenate(
            [up.cpu().numpy(), vp.cpu().numpy()], 1), axis=1)
        umag_true = np.linalg.norm(np.stack([gt["u"], gt["v"]], 1), axis=1)

        fig, ax = plt.subplots(3, 1, figsize=(11, 7), constrained_layout=True)
        for a, f, t in [(ax[0], umag_true, "|u| ground truth (FEM)"),
                        (ax[1], umag_pred, "|u| PINN prediction"),
                        (ax[2], np.abs(umag_pred - umag_true), "|error|")]:
            sc = a.scatter(gt["x"], gt["y"], c=f, s=2, cmap="viridis")
            a.set_aspect("equal"); a.set_title(t)
            a.set_xlim(0, cfg.L); a.set_ylim(0, cfg.H)
            fig.colorbar(sc, ax=a, fraction=0.02)
        fig.savefig(os.path.join(outdir, "pinn_cylinder.png"), dpi=130)
        plt.close(fig)
        print(f"wrote {os.path.join(outdir, 'pinn_cylinder.png')}")
    except Exception as e:
        print(f"(plot skipped: {e})")


if __name__ == "__main__":
    train()

loaded ground truth: Re=20.0, 23756 points
[adam     0] loss=5.688e-01  pde=1.09e-01 bc=4.09e-02 gauge=4.26e-04 data=1.01e-02
[adam   500] loss=1.471e-01  pde=1.44e-02 bc=7.83e-03 gauge=1.33e-05 data=1.09e-02
[adam  1000] loss=1.087e-01  pde=1.87e-02 bc=6.30e-03 gauge=1.06e-05 data=5.39e-03
[adam  1500] loss=4.381e-02  pde=8.86e-03 bc=1.86e-03 gauge=7.26e-06 data=3.27e-03
[adam  1999] loss=3.070e-02  pde=6.04e-03 bc=1.40e-03 gauge=2.58e-06 data=2.13e-03
[lbfgs done] loss=3.108e-02  pde=7.14e-03 bc=1.23e-03 data=2.32e-03  (1672.5s)
velocity relative L2 vs FEM ground truth: 3.6790e-01
wrote pinn_out\pinn_cylinder.png
